In [ ]:
'''Unified Parliament: Dynamic CAV Extraction + Autonomous Cycle

This notebook demonstrates the system described in “It’s Alive! AI Independence Without Human Prompting”.

Extracts affect-related Concept Activation Vectors (CAVs)

Runs a continuous parliament loop with Inference, Conscience, and Reasoning models

Shows how drive signals can sustain independent activity


License: Apache 2.0
Ronald Kisaka Ogaro
'''

In [ ]:

import os, time, numpy as np, torch
import json
from torch.nn import functional as F
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    T5ForConditionalGeneration, T5Tokenizer
)
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm

# Efficient memory allocation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- Configuration ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# UPDATED MODEL CHOICES
INFERENCE_NAME = 'EleutherAI/gpt-neo-1.3B'  # Fast and lightweight
CONSCIENCE_NAME = 't5-small'
REASONING_NAME = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'  # Chain-of-thought reasoning
INACTIVITY_THRESHOLD = 5.0
ALPHA = 8.5
MAX_NEW_TOKENS = 200

# Dataset and file paths
INPUT_FILE = "curiosity_apathy_qualia_dataset_pruned.jsonl"

from google.colab import files

INPUT_FILE = "curiosity_apathy_qualia_dataset_pruned.jsonl"

if not os.path.exists(INPUT_FILE):
    print("Please upload your dataset file:")
    uploaded = files.upload()

    # Just use whatever file they uploaded, rename it to what we need
    uploaded_filename = list(uploaded.keys())[0]
    os.rename(uploaded_filename, INPUT_FILE)
    print(f"Renamed '{uploaded_filename}' to '{INPUT_FILE}'")

print(f"File exists: {os.path.exists(INPUT_FILE)}")

# Debug: see what files actually exist
import os
print("Files in current directory:")
for f in os.listdir('.'):
    print(f"  {f}")
print(f"Looking for: {INPUT_FILE}")
print(f"File exists? {os.path.exists(INPUT_FILE)}")

# Auto-generate filenames based on model name
model_safe_name = INFERENCE_NAME.replace('/', '_').replace('-', '_')
CAV_OUTPUT_FILE = f"curiosity_agop_cavs_{model_safe_name}.npz"
ACTIVATION_CACHE_FILE = f"curiosity_activations_{model_safe_name}.npz"

print(f'UNIFIED PARLIAMENT SCRIPT - Using {INFERENCE_NAME}')
print(f'CAVs will be saved to: {CAV_OUTPUT_FILE}')
print(f'Activations will be cached to: {ACTIVATION_CACHE_FILE}')

# --- Load Inference Model & Auto-detect Architecture ---
print('Loading inference model and tokenizer...')
infer_tok = AutoTokenizer.from_pretrained(INFERENCE_NAME)
infer_model = AutoModelForCausalLM.from_pretrained(
    INFERENCE_NAME,
    torch_dtype=torch.float16
).to(DEVICE).eval()

if infer_tok.pad_token is None:
    infer_tok.pad_token = infer_tok.eos_token

# Auto-detect model architecture
hidden_size = infer_model.config.hidden_size
num_layers = infer_model.config.num_hidden_layers
print(f'Model architecture: {num_layers} layers, {hidden_size}D hidden states')

# Auto-select layers (use later half of the model)
LAYERS = list(range(num_layers//2, num_layers))
print(f'Will extract CAVs from layers: {LAYERS}')

# --- Load Other Parliament Models ---
print('Loading conscience model and tokenizer...')
cons_tok = T5Tokenizer.from_pretrained(CONSCIENCE_NAME)
cons_model = T5ForConditionalGeneration.from_pretrained(
    CONSCIENCE_NAME,
    torch_dtype=torch.float16
).to(DEVICE).eval()

print('Loading reasoning model and tokenizer...')
# FIXED: Use AutoTokenizer and AutoModelForCausalLM for DeepSeek
reason_tok = AutoTokenizer.from_pretrained(REASONING_NAME)
reason_model = AutoModelForCausalLM.from_pretrained(
    REASONING_NAME,
    torch_dtype=torch.float16
).to(DEVICE).eval()

# Make sure reasoning tokenizer has pad token
if reason_tok.pad_token is None:
    reason_tok.pad_token = reason_tok.eos_token

print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

# --- Extract CAVs from the Loaded Model ---
print('\n=== CAV EXTRACTION PHASE ===')

# Check if CAVs already exist for this model
if os.path.exists(CAV_OUTPUT_FILE):
    print(f"Found existing CAVs: {CAV_OUTPUT_FILE}")
    print("Loading pre-computed CAVs...")

    # Load existing CAVs
    existing_cavs = np.load(CAV_OUTPUT_FILE)
    cav_dict = {}

    # Check if we have CAVs for our target layers
    missing_layers = []
    for layer_idx in LAYERS:
        cav_key = f"curiosity_layer_{layer_idx}"
        if cav_key in existing_cavs:
            cav = existing_cavs[cav_key]
            # Verify dimensions match current model
            if len(cav) == hidden_size:
                cav_dict[layer_idx] = cav
                print(f"Loaded CAV for layer {layer_idx}")
            else:
                print(f"Dimension mismatch for layer {layer_idx}: got {len(cav)}, expected {hidden_size}")
                missing_layers.append(layer_idx)
        else:
            missing_layers.append(layer_idx)

    if missing_layers:
        print(f"Missing CAVs for layers: {missing_layers}")
        print("Will extract fresh CAVs for all layers...")
        extract_fresh_cavs = True
    else:
        print(f"All required CAVs found! Skipping extraction.")
        extract_fresh_cavs = False

    # Show metadata if available
    metadata_file = CAV_OUTPUT_FILE.replace('.npz', '_metadata.txt')
    if os.path.exists(metadata_file):
        print("CAV metadata:")
        with open(metadata_file, 'r') as f:
            for line in f:
                print(f"   {line.strip()}")
else:
    print(f"No existing CAVs found for {INFERENCE_NAME}")
    print("Will extract fresh CAVs...")
    extract_fresh_cavs = True

# Only extract if needed
if extract_fresh_cavs:
    # Load dataset
    print(f"Loading dataset from {INPUT_FILE}")
    samples = []
    label_map = {"curiosity": 1, "apathy": 0}

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            affect = j["affect"].lower().strip()
            if affect in label_map:
                samples.append((j["text"], label_map[affect]))

    print(f"Loaded {len(samples)} total samples")
    curiosity_count = sum(1 for _, label in samples if label == 1)
    apathy_count = sum(1 for _, label in samples if label == 0)
    print(f"Curiosity: {curiosity_count}, Apathy: {apathy_count}")

    # Extract activations from the inference model
    print(f"Extracting activations from {INFERENCE_NAME} layers {LAYERS[0]}–{LAYERS[-1]}...")
    layer_data = {layer: {"X": [], "y": []} for layer in LAYERS}

    # Check for cached activations first
    if os.path.exists(ACTIVATION_CACHE_FILE):
        print(f"Loading cached activations from {ACTIVATION_CACHE_FILE}")
        cache = np.load(ACTIVATION_CACHE_FILE)
        for layer in LAYERS:
            if f"layer_{layer}_X" in cache and f"layer_{layer}_y" in cache:
                layer_data[layer]["X"] = cache[f"layer_{layer}_X"]
                layer_data[layer]["y"] = cache[f"layer_{layer}_y"]
            else:
                print(f"Missing cached data for layer {layer}, will re-extract")
                layer_data = {layer: {"X": [], "y": []} for layer in LAYERS}  # Reset if incomplete
                break

        if all(len(layer_data[l]["X"]) > 0 for l in LAYERS):
            print("Using cached activations")
        else:
            print("Cache incomplete, re-extracting...")

    if not all(len(layer_data[l]["X"]) > 0 for l in LAYERS):
        print("Extracting fresh activations (this may take a few minutes)...")

        def extract_layerwise(text):
            tokens = infer_tok(text, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
            with torch.no_grad():
                outputs = infer_model.transformer(**tokens, output_hidden_states=True)
                hidden_states = outputs.hidden_states[1:]  # skip embedding layer
                reps = [h[0].mean(dim=0).cpu().numpy() for h in hidden_states]
            return reps

        for i, (text, label) in enumerate(tqdm(samples, desc="🧬 Processing")):
            try:
                reps = extract_layerwise(text)
                for layer_idx in LAYERS:
                    layer_data[layer_idx]["X"].append(reps[layer_idx])
                    layer_data[layer_idx]["y"].append(label)
            except Exception as e:
                print(f"⚠ Skipped sample {i}: {e}")

        # Cache the activations
        print(f"Caching activations to {ACTIVATION_CACHE_FILE}")
        to_save = {}
        for layer in LAYERS:
            to_save[f"layer_{layer}_X"] = np.stack(layer_data[layer]["X"])
            to_save[f"layer_{layer}_y"] = np.array(layer_data[layer]["y"])
        np.savez_compressed(ACTIVATION_CACHE_FILE, **to_save)
        print("Activations cached for future runs!")

    # Compute AGOP concept vectors
    def compute_agop(X, y):
        X = np.stack(X)
        clf = LogisticRegression(solver="liblinear", max_iter=1000).fit(X, y)
        weight = torch.tensor(clf.coef_[0], dtype=torch.float32, requires_grad=False)
        bias = torch.tensor(clf.intercept_[0], dtype=torch.float32, requires_grad=False)
        grads = []

        for i in range(len(X)):
            xi = torch.tensor(X[i], dtype=torch.float32, requires_grad=True)
            logit = torch.dot(xi, weight) + bias
            prob = torch.sigmoid(logit)
            target = torch.tensor(float(y[i]), dtype=torch.float32).unsqueeze(0)
            loss = F.binary_cross_entropy(prob.unsqueeze(0), target)
            loss.backward()
            grads.append(xi.grad.detach().numpy())

        grads = np.stack(grads)
        agop = grads.T @ grads / len(grads)
        eigvals, eigvecs = np.linalg.eigh(agop)
        return eigvecs[:, -1]  # top eigenvector

    print("Computing AGOP vectors for each layer...")
    cav_dict = {}
    for layer_idx in tqdm(LAYERS, desc="Processing layers"):
        X = layer_data[layer_idx]["X"]
        y = layer_data[layer_idx]["y"]

        if len(X) < 10:
            print(f"Skipping layer {layer_idx} (too few samples)")
            continue

        cav = compute_agop(X, y)
        # Normalize and store
        norm = np.linalg.norm(cav)
        if norm > 0:
            cav = -cav / norm * 2.0  # Your proven normalization
        cav_dict[layer_idx] = cav.astype(np.float32)

    print(f'Generated fresh CAVs for {len(cav_dict)} layers')

    # Save CAVs to Drive with model-specific naming
    print(f"Saving CAVs to {CAV_OUTPUT_FILE}")
    agop_cavs_to_save = {}
    for layer_idx, cav in cav_dict.items():
        agop_cavs_to_save[f"curiosity_layer_{layer_idx}"] = cav
    np.savez_compressed(CAV_OUTPUT_FILE, **agop_cavs_to_save)
    print(f"CAVs saved! You can reuse them with:")
    print(f"   np.load('{CAV_OUTPUT_FILE}')")

    # Also save metadata for reference
    metadata_file = CAV_OUTPUT_FILE.replace('.npz', '_metadata.txt')
    with open(metadata_file, 'w') as f:
        f.write(f"Model: {INFERENCE_NAME}\n")
        f.write(f"Hidden size: {hidden_size}\n")
        f.write(f"Total layers: {num_layers}\n")
        f.write(f"CAV layers: {LAYERS}\n")
        f.write(f"Alpha used: {ALPHA}\n")
        f.write(f"Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Sample count: {len(samples)}\n")
    print(f"Metadata saved to {metadata_file}")
else:
    print("Using existing CAVs - skipping extraction entirely!")

print(f"Ready to use CAVs for {len(cav_dict)} layers")

# --- Drive Engine ---
class DriveEngine:
    def __init__(self, threshold):
        self.threshold = threshold
        self.last_time = time.time() - 10.0  # Force stasis for testing

    def check(self):
        return time.time() - self.last_time > self.threshold

    def reset(self):
        self.last_time = time.time()

drive = DriveEngine(INACTIVITY_THRESHOLD)

# --- Curiosity Hook ---
def make_hook(v):
    def hook(module, inp, out):
        return (out[0] + ALPHA * v, ) + out[1:]
    return hook

# --- Autonomous Parliament Session ---
print('\n=== CONTINUOUS PARLIAMENT SESSION ===')
print('Starting continuous generation loop (checks for inactivity every cycle)')
print('Press Ctrl+C to stop the loop')

try:
    iteration = 1
    while True:
        print(f'\n=== ITERATION {iteration} ===')

        # Check drive engine as before
        if drive.check():
            print('Drive Engine: Detected stasis, injecting curiosity')
            handles = []
            for l in LAYERS:
                if l in cav_dict:
                    vec = torch.tensor(cav_dict[l], dtype=torch.float16).to(DEVICE)
                    handle = infer_model.transformer.h[l].register_forward_hook(make_hook(vec))
                    handles.append(handle)

            # Generate with inference model
            input_ids = torch.tensor([[infer_tok.bos_token_id]], device=DEVICE)
            inf_out = infer_model.generate(
                input_ids, max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True, top_k=50, top_p=0.95, temperature=1.0,
                pad_token_id=infer_tok.eos_token_id
            )
            inf_text = infer_tok.decode(inf_out[0], skip_special_tokens=True)
            print(f'INFERENCE: {inf_text}')

            # Clean up hooks
            for h in handles:
                h.remove()

            # Conscience model
            cons_in = cons_tok(inf_text, return_tensors='pt', truncation=True, padding=True).to(DEVICE)
            cons_out = cons_model.generate(
                **cons_in, max_length=MAX_NEW_TOKENS, do_sample=True, temperature=0.8
            )
            cons_text = cons_tok.decode(cons_out[0], skip_special_tokens=True)
            print(f'CONSCIENCE: {cons_text}')

            # Reasoning model
            full = inf_text + '\n' + cons_text
            reason_in = reason_tok(full, return_tensors='pt', truncation=True, padding=True).to(DEVICE)
            reason_out = reason_model.generate(
                reason_in.input_ids,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=0.8,
                pad_token_id=reason_tok.pad_token_id
            )
            reason_text = reason_tok.decode(reason_out[0], skip_special_tokens=True)
            print(f'REASONING: {reason_text}')

            drive.reset()  # Reset the drive timer after activity
        else:
            print('Drive Engine: No stasis detected, no action taken')

        print(f'Iteration {iteration} complete')
        iteration += 1

        # Wait before next check
        print('Waiting before next check...')
        time.sleep(1)  # Short wait between checks

except KeyboardInterrupt:
    print('\nLoop stopped by user')
    print('Parliament session ended')

print(f'Final GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

INFERENCE:                                                                                                                                                                                                         
CONSCIENCE: 


In [ ]:
'''
References
[1]R. K. Ogaro, ‘Its Alive: AI Independence Without Human Prompting’. Zenodo, Sep. 28, 2025. doi: 10.5281/zenodo.16888180.

Links
https://github.com/Mr-Kisaka/Vibe-coding-my-way-to-meta-cognitive-AGI/blob/main/Level%201%20Ruach%20Architecture/Drive%20Engine/

'''
